# Example: Reviewing a Captured Autotrader Decision Queue
In this example, we trace proposed portfolio changes through deterministic risk and escalation gates using a captured event table rather than live brokerage or language-model services.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Apply pre-trade controls:__ Test position, turnover, drawdown, and news-severity limits.
> * __Route decisions transparently:__ Distinguish approved, blocked, and human-review outcomes.
> * __Build an auditable ticket:__ Preserve proposed actions, gate reasons, and reviewer decisions in a compact record.

Let's operate the teaching system without hiding its decisions behind a live API.
___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading the packages used in this example.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. The file activates the course environment, defines notebook-relative paths, and loads the required packages.

Let's set up the code environment:

The reusable portfolio algorithms in this example are provided by the local [`VLQuantitativeFinancePackage.jl`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/) package.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


For additional information, see the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

___


## Task 1: Load the Captured Decision Queue
The table represents one evaluation interval. `news_severity` measures the magnitude of a captured information event, not whether the event is favorable or unfavorable.


In [ ]:
events = DataFrame(
    timestamp=["10:00", "10:00", "10:00", "10:00", "10:00"],
    ticker=["ALFA", "BRAV", "CHAR", "DELT", "ECHO"],
    current_weight=[0.18, 0.22, 0.20, 0.16, 0.24],
    proposed_weight=[0.14, 0.31, 0.10, 0.17, 0.28],
    news_severity=[0.20, 0.35, 0.88, 0.10, 0.79],
    signal=[-0.15, 0.42, -0.70, 0.08, 0.51],
);
portfolio_drawdown = 0.075;
limits = (
    max_position_weight=0.30,
    max_one_way_turnover=0.20,
    drawdown_escalation=0.10,
    news_severity_escalation=0.75,
);


## Task 2: Apply the Gate
A position-limit breach blocks the proposed action. A loud-news event or portfolio drawdown breach routes it to human review. Otherwise, the action is approved. Portfolio turnover is evaluated across the complete proposed allocation.


In [ ]:
one_way_turnover = 0.5*sum(abs.(events.proposed_weight .- events.current_weight));
turnover_review = one_way_turnover > limits.max_one_way_turnover;

routing = [route_portfolio_event(row, portfolio_drawdown, limits, turnover_review)
    for row in eachrow(events)];
events.route = getproperty.(routing, :route);
events.reason = getproperty.(routing, :reason);
pretty_table(events; table_format=TextTableFormat(borders=text_table_borders__simple))


## Task 3: Build the Review Ticket
For classroom use, the reviewer approves ordinary actions, rejects blocked actions, and modifies review-routed actions by retaining the current weight. The ticket records both the engine proposal and the final authorized weight.


In [ ]:
review_decision = Dict(
    "APPROVE" => "ACCEPT",
    "BLOCK" => "REJECT",
    "REVIEW" => "HOLD_CURRENT",
);

ticket = select(events, :timestamp, :ticker, :current_weight, :proposed_weight,
    :route, :reason);
ticket.decision = [review_decision[r] for r in ticket.route];
ticket.authorized_weight = [
    ticket.decision[i] == "ACCEPT" ? ticket.proposed_weight[i] : ticket.current_weight[i]
    for i in 1:nrow(ticket)
];

pretty_table(ticket; table_format=TextTableFormat(borders=text_table_borders__simple));
println("Proposed one-way turnover: $(round(one_way_turnover, digits=4))")


## Final Challenge Extension
Replace the captured proposal with the output of your Autotrader, but retain the gate interface and evidence table. Your submission must preserve:

1. the inputs visible to the strategy;
2. the proposed allocation;
3. every gate result and reason;
4. every human modification;
5. the final authorized allocation;
6. the benchmark and risk scorecard used in the defense.

The objective is not to eliminate human judgment. It is to make the boundary between automated and human judgment explicit and auditable.


## Summary
This example demonstrated a credential-free operational control loop using captured inputs.

> __Key Takeaways:__
>
> * __Severity is not direction:__ A loud event can require review whether its sentiment is positive or negative.
> * __Controls need explicit routing behavior:__ A limit should state whether it approves, blocks, or escalates an action.
> * __The decision record is part of the system:__ Proposed and authorized allocations must remain distinguishable.

The final course challenge extends this same interface to each team's strategy.
___

## Disclaimer and Risks
This material is for educational purposes only. The example does not connect to a brokerage, submit orders, or constitute a complete compliance system.
